# Notebook 3 — Elegir el mejor modelo

**Objetivo**: decidir con qué familia de modelo predecir goles (Poisson GLM, LightGBM o
XGBoost), y hacerlo con evidencia sólida, no con el resultado de un solo torneo. Antes este
notebook solo comparaba contra la fase de grupos de 2026 -- un bootstrap sobre esos 72
partidos da un intervalo de confianza de RMSE de casi 0.4 puntos, mucho más ancho que la
diferencia real entre familias. Un solo torneo no basta para decidir.

**Aquí se entrena y evalúa la misma comparación 5 veces**: los Mundiales de 2010, 2014, 2018
y 2022 (entrenando cada vez solo con datos anteriores a su inicio, sin verlos nunca de
antemano) más 2026. La familia que gane consistentemente en los 5 -- no solo en uno -- es la
que de verdad conviene usar en el Notebook 4.

Random Forest no participa: en una comparación previa perdió con claridad y era, con
diferencia, el más lento de tunear (~48 min por torneo) -- no aporta señal que justifique
repetirlo 5 veces.

Entrada: `data/processed/partidos_features.csv` (Notebook 2).
Salida: `results/comparacion_modelos.csv` (detalle por Mundial y familia),
`results/comparacion_resumen.csv` (ranking agregado).

In [1]:
import time
from pathlib import Path

import lightgbm as lgb
import networkx as nx
import numpy as np
import optuna
import pandas as pd
import xgboost as xgb
from scipy.stats import poisson
from sklearn.base import clone
from sklearn.linear_model import PoissonRegressor
from sklearn.metrics import log_loss, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

optuna.logging.set_verbosity(optuna.logging.WARNING)

DIR_RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = DIR_RAIZ / "data" / "processed"
DIR_RESULTS = DIR_RAIZ / "results"
DIR_RESULTS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 25)
pd.set_option("display.width", 160)

SEMILLA = 42
FECHA_CORTE_HISTORICO = pd.Timestamp("1990-01-01")
FEATURES_MODELO = [
    "elo_diff", "tendencia_elo_local", "tendencia_elo_visitante",
    "dif_forma_gf_5", "dif_forma_gf_10", "dif_racha_5", "dif_racha_10",
    "dias_descanso_local", "dias_descanso_visitante",
    "es_neutral", "h2h_puntos_prom", "h2h_dif_goles_prom",
    # Palmarés reciente (Notebook 2, sección 2.6b) -- adoptadas tras ganar en el
    # backtest de 5 Mundiales y confirmarse en 82 ediciones (ver registro al final).
    "dif_titulos_8a", "dif_titulos_4a", "dif_finales_8a", "dif_ppp_vs_top_3a",
    # Valor de mercado de plantilla (2.6c) -- mejor evidencia agregada del
    # proyecto: gana en RMSE/LogLoss en la mayoría de 33 ediciones 2010-2026.
    "dif_valor_plantilla",
]
MUNDIALES = [2010, 2014, 2018, 2022, 2026]  # los 5 torneos disponibles en la era moderna

df = pd.read_csv(DIR_PROCESSED / "partidos_features.csv", parse_dates=["fecha"])
df = df[df["fecha"] >= FECHA_CORTE_HISTORICO].reset_index(drop=True)
print(f"Partidos desde {FECHA_CORTE_HISTORICO.date()}: {len(df):,}")

Partidos desde 1990-01-01: 32,382


## 3.1 Fase de grupos de cada Mundial, entrenamiento estrictamente anterior

Camarillas de 4 en el grafo de enfrentamientos: funciona igual para el formato de 32 equipos
(2010-2022) que para el de 48/12 grupos de 2026. Para cada uno de los 5 torneos, el
entrenamiento es SOLO lo anterior a su fecha de inicio -- ninguno ve su propio resultado, ni
siquiera 2026 (que además es el único con eliminatoria ya jugada, pero aquí solo se usa su
fase de grupos, igual que los demás).

In [2]:
def detectar_fase_grupos(df_mundial: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    grafo = nx.Graph()
    for _, fila in df_mundial.iterrows():
        grafo.add_edge(fila["equipo_local"], fila["equipo_visitante"])

    camarillas_de_4 = [c for c in nx.enumerate_all_cliques(grafo) if len(c) == 4]
    usados: set[str] = set()
    grupos: list[frozenset[str]] = []
    for c in camarillas_de_4:
        if not (set(c) & usados):
            grupos.append(frozenset(c))
            usados.update(c)

    def _mismo_grupo(fila) -> bool:
        par = {fila["equipo_local"], fila["equipo_visitante"]}
        return any(par <= g for g in grupos)

    es_fase_grupos = df_mundial.apply(_mismo_grupo, axis=1)
    return df_mundial[es_fase_grupos].copy(), df_mundial[~es_fase_grupos].copy()


splits_por_torneo = {}
for year in MUNDIALES:
    mundial = df[(df["torneo"] == "FIFA World Cup") & (df["fecha"].dt.year == year)]
    grupos, _eliminatoria = detectar_fase_grupos(mundial)
    fecha_inicio = mundial["fecha"].min()
    train = df[(df["fecha"] < fecha_inicio) & df["jugado"]].copy()
    test = grupos[grupos["jugado"]].copy()
    splits_por_torneo[year] = {"train": train, "test": test, "fecha_inicio": fecha_inicio}
    print(f"Mundial {year} (inicio {fecha_inicio.date()}): entrenamiento {len(train):,} partidos, "
          f"test (fase de grupos) {len(test)} partidos.")

Mundial 2010 (inicio 2010-06-11): entrenamiento 16,746 partidos, test (fase de grupos) 48 partidos.
Mundial 2014 (inicio 2014-06-12): entrenamiento 20,743 partidos, test (fase de grupos) 48 partidos.
Mundial 2018 (inicio 2018-06-14): entrenamiento 24,521 partidos, test (fase de grupos) 49 partidos.
Mundial 2022 (inicio 2022-11-20): entrenamiento 28,582 partidos, test (fase de grupos) 49 partidos.
Mundial 2026 (inicio 2026-06-11): entrenamiento 32,286 partidos, test (fase de grupos) 72 partidos.


## 3.2 Métrica común: RMSE de goles + LogLoss 1X2, con Dixon-Coles

Una sola implementación para las cinco comparaciones -- antes esta parte estaba duplicada
entre "comparar en 2026" y "backtestear en Mundiales pasados", y las dos versiones habían
divergido: la de 2026 nunca aplicaba la corrección de Dixon-Coles al LogLoss (una
inconsistencia real, no intencionada). Aquí solo hay una versión, con Dixon-Coles siempre
aplicado, y cada torneo calibra su propio `rho` sobre SU PROPIO entrenamiento (calibrarlo con
datos posteriores al torneo evaluado sería la misma fuga temporal que el resto del pipeline
evita en todo lo demás).

In [3]:
def log_verosimilitud_dixon_coles(rho: float, lam: np.ndarray, mu: np.ndarray, x: np.ndarray, y: np.ndarray) -> float:
    tau = np.ones_like(lam)
    es_0_0 = (x == 0) & (y == 0)
    es_0_1 = (x == 0) & (y == 1)
    es_1_0 = (x == 1) & (y == 0)
    es_1_1 = (x == 1) & (y == 1)
    tau[es_0_0] = 1 - lam[es_0_0] * mu[es_0_0] * rho
    tau[es_0_1] = 1 + lam[es_0_1] * rho
    tau[es_1_0] = 1 + mu[es_1_0] * rho
    tau[es_1_1] = 1 - rho
    tau = np.clip(tau, 1e-8, None)
    return float((np.log(tau) + poisson.logpmf(x, lam) + poisson.logpmf(y, mu)).sum())


def ajustar_rho_dixon_coles(modelos: dict, X: pd.DataFrame, goles_local: pd.Series, goles_visitante: pd.Series) -> float:
    lam = np.clip(modelos["local"].predict(X), 0.01, None)
    mu = np.clip(modelos["visitante"].predict(X), 0.01, None)
    x, y = goles_local.to_numpy(), goles_visitante.to_numpy()
    candidatos_rho = np.linspace(-0.3, 0.3, 121)
    log_verosimilitudes = [log_verosimilitud_dixon_coles(r, lam, mu, x, y) for r in candidatos_rho]
    return float(candidatos_rho[np.argmax(log_verosimilitudes)])


def probabilidades_1x2_desde_lambdas(lam_local: np.ndarray, lam_visitante: np.ndarray, rho: float,
                                       max_goles: int = 10) -> np.ndarray:
    goles = np.arange(max_goles + 1)
    salida = np.zeros((len(lam_local), 3))
    for i, (la, lv) in enumerate(zip(lam_local, lam_visitante)):
        rejilla = np.outer(poisson.pmf(goles, la), poisson.pmf(goles, lv))
        rejilla[0, 0] *= 1 - la * lv * rho
        rejilla[0, 1] *= 1 + la * rho
        rejilla[1, 0] *= 1 + lv * rho
        rejilla[1, 1] *= 1 - rho
        rejilla = np.clip(rejilla, 0, None)
        salida[i] = [np.tril(rejilla, -1).sum(), np.trace(rejilla), np.triu(rejilla, 1).sum()]
    return salida / salida.sum(axis=1, keepdims=True)


def evaluar_modelo_goles(modelos: dict, X: pd.DataFrame, df_partidos: pd.DataFrame, rho: float) -> dict:
    pred_local = np.clip(modelos["local"].predict(X), 0.01, None)
    pred_visitante = np.clip(modelos["visitante"].predict(X), 0.01, None)

    rmse = np.sqrt(mean_squared_error(
        pd.concat([df_partidos["goles_local"], df_partidos["goles_visitante"]]),
        np.concatenate([pred_local, pred_visitante]),
    ))
    probs = probabilidades_1x2_desde_lambdas(pred_local, pred_visitante, rho)
    y_real = df_partidos["resultado_1x2"].map({"LOCAL": 0, "EMPATE": 1, "VISITANTE": 2}).values
    logloss = log_loss(y_real, probs, labels=[0, 1, 2])

    errores_cuadrados = np.concatenate([
        (pred_local - df_partidos["goles_local"].to_numpy()) ** 2,
        (pred_visitante - df_partidos["goles_visitante"].to_numpy()) ** 2,
    ])
    return {"rmse": rmse, "logloss": logloss, "errores_cuadrados": errores_cuadrados}


def rmse_cv_temporal(estimador, X: pd.DataFrame, y: pd.Series, n_splits: int = 5) -> float:
    cv = TimeSeriesSplit(n_splits=n_splits)
    rmses = []
    for idx_train, idx_val in cv.split(X):
        modelo = clone(estimador).fit(X.iloc[idx_train], y.iloc[idx_train])
        pred = np.clip(modelo.predict(X.iloc[idx_val]), 0, None)
        rmses.append(np.sqrt(mean_squared_error(y.iloc[idx_val], pred)))
    return float(np.mean(rmses))


def optimizar_familia(construir_estimador, X: pd.DataFrame, y: pd.Series, n_trials: int, nombre: str) -> optuna.Study:
    inicio = time.time()
    estudio = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEMILLA))
    estudio.optimize(lambda trial: rmse_cv_temporal(construir_estimador(trial), X, y),
                      n_trials=n_trials, show_progress_bar=False)
    print(f"  [{nombre}] RMSE CV: {estudio.best_value:.4f} ({time.time() - inicio:.0f}s) | {estudio.best_params}")
    return estudio

## 3.3 Espacio de hiperparámetros (GLM, LightGBM, XGBoost)

Para el GLM, un barrido logarítmico de `alpha` (regularización L2) -- antes se dejaba fijo en
0.5 sin haberlo probado nunca contra otro valor. Para los árboles/boosting, los mismos rangos
que ya se habían validado.

In [4]:
def construir_glm(trial: optuna.Trial):
    alpha = trial.suggest_float("alpha", 1e-3, 100.0, log=True)
    return make_pipeline(StandardScaler(), PoissonRegressor(alpha=alpha, max_iter=1000))


def construir_lightgbm(trial: optuna.Trial) -> lgb.LGBMRegressor:
    return lgb.LGBMRegressor(
        objective="poisson", random_state=SEMILLA, verbosity=-1,
        n_estimators=trial.suggest_int("n_estimators", 100, 300),
        num_leaves=trial.suggest_int("num_leaves", 8, 64),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 100),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
    )


def construir_xgboost(trial: optuna.Trial) -> xgb.XGBRegressor:
    return xgb.XGBRegressor(
        objective="count:poisson", random_state=SEMILLA, verbosity=0,
        n_estimators=trial.suggest_int("n_estimators", 100, 300),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.6, 1.0),
    )


FAMILIAS = {"glm": construir_glm, "lightgbm": construir_lightgbm, "xgboost": construir_xgboost}
TRIALS_POR_FAMILIA = {"glm": 20, "lightgbm": 15, "xgboost": 15}

## 3.4 Afinar y evaluar cada familia, en cada uno de los 5 Mundiales

Mismo procedimiento repetido 5 veces: afinar las 3 familias SOLO con datos anteriores al
inicio de ese torneo, calibrar `rho` sobre ese mismo entrenamiento, y evaluar sobre su fase de
grupos real. Se guardan también los errores cuadrado a cuadrado, para el bootstrap de 3.5.

In [5]:
resultados = []
errores_por_torneo_familia = {}

for year in MUNDIALES:
    train = splits_por_torneo[year]["train"]
    test = splits_por_torneo[year]["test"]
    X_train, X_test = train[FEATURES_MODELO], test[FEATURES_MODELO]
    print(f"\n=== Mundial {year} ({len(train):,} partidos de entrenamiento, {len(test)} de test) ===")

    modelos_glm_temp = {
        "local": make_pipeline(StandardScaler(), PoissonRegressor(alpha=0.2, max_iter=1000)).fit(X_train, train["goles_local"]),
        "visitante": make_pipeline(StandardScaler(), PoissonRegressor(alpha=0.2, max_iter=1000)).fit(X_train, train["goles_visitante"]),
    }
    rho_torneo = ajustar_rho_dixon_coles(modelos_glm_temp, X_train, train["goles_local"], train["goles_visitante"])
    print(f"  rho Dixon-Coles: {rho_torneo:.4f}")

    for nombre_familia, construir in FAMILIAS.items():
        n_trials = TRIALS_POR_FAMILIA[nombre_familia]
        estudio_local = optimizar_familia(construir, X_train, train["goles_local"], n_trials, f"{nombre_familia}/local")
        estudio_visitante = optimizar_familia(construir, X_train, train["goles_visitante"], n_trials, f"{nombre_familia}/visitante")

        modelos = {
            "local": construir(optuna.trial.FixedTrial(estudio_local.best_params)).fit(X_train, train["goles_local"]),
            "visitante": construir(optuna.trial.FixedTrial(estudio_visitante.best_params)).fit(X_train, train["goles_visitante"]),
        }
        metricas = evaluar_modelo_goles(modelos, X_test, test, rho_torneo)
        errores_por_torneo_familia[(year, nombre_familia)] = metricas.pop("errores_cuadrados")
        resultados.append({"mundial": year, "familia": nombre_familia, **metricas})
        rmse_v, logloss_v = metricas["rmse"], metricas["logloss"]
        print(f"  [{nombre_familia}] RMSE test: {rmse_v:.4f}  |  LogLoss test: {logloss_v:.4f}")

df_comparacion = pd.DataFrame(resultados)
print("\n=== Resultado por Mundial y familia ===")
print(df_comparacion.pivot(index="mundial", columns="familia", values="rmse").round(4).to_string())


=== Mundial 2010 (16,746 partidos de entrenamiento, 48 de test) ===


  rho Dixon-Coles: -0.0550


  [glm/local] RMSE CV: 1.6481 (1s) | {'alpha': 2.3774410482619617}


  [glm/visitante] RMSE CV: 1.2439 (1s) | {'alpha': 0.18191071723197713}
  [glm] RMSE test: 1.2097  |  LogLoss test: 1.0206


  [lightgbm/local] RMSE CV: 1.6036 (27s) | {'n_estimators': 171, 'num_leaves': 32, 'learning_rate': 0.036928006593336324, 'min_child_samples': 36, 'reg_lambda': 0.8389938644332399, 'subsample': 0.6102570838225161, 'colsample_bytree': 0.8953953954952381}


...

  [lightgbm/visitante] RMSE CV: 1.2217 (23s) | {'n_estimators': 180, 'num_leaves': 38, 'learning_rate': 0.03675548837009079, 'min_child_samples': 38, 'reg_lambda': 0.32992221734335486, 'subsample': 0.6012391969272454, 'colsample_bytree': 0.8847911208853647}


  [lightgbm] RMSE test: 1.2977  |  LogLoss test: 0.8931


  [xgboost/local] RMSE CV: 1.5269 (16s) | {'n_estimators': 299, 'max_depth': 5, 'learning_rate': 0.037155275845518615, 'min_child_weight': 5, 'reg_lambda': 0.23079754987887754, 'subsample': 0.9069820711004439, 'colsample_bytree': 0.7404885473672256}


  [xgboost/visitante] RMSE CV: 1.2229 (12s) | {'n_estimators': 238, 'max_depth': 3, 'learning_rate': 0.04950738234554274, 'min_child_weight': 4, 'reg_lambda': 0.005284852659147412, 'subsample': 0.9954992856822716, 'colsample_bytree': 0.8847911208853647}


  [xgboost] RMSE test: 1.2812  |  LogLoss test: 0.8715

=== Resultado por Mundial y familia ===
familia     glm  lightgbm  xgboost
mundial                           
2010     1.2097    1.0834   1.0847
2014     1.2624    1.2253   1.2346
2018     1.2086    1.1621   1.1830
2022     1.3389    1.2530   1.2512
2026     1.2574    1.2977   1.2812


## 3.5 Bootstrap por torneo: ¿la diferencia entre familias es ruido?

Si el intervalo de confianza del RMSE de la familia ganadora se solapa con el de las demás,
la victoria de ESE torneo concreto no es concluyente por sí sola -- por eso hace falta mirar
los 5 juntos (3.6), no solo confiar en cuál "gana" en cada uno por separado.

In [6]:
def bootstrap_ci_rmse(errores_cuadrados: np.ndarray, n_boot: int = 2000, semilla: int = SEMILLA) -> tuple[float, float]:
    rng = np.random.default_rng(semilla)
    n = len(errores_cuadrados)
    remuestreos = rng.integers(0, n, size=(n_boot, n))
    rmses_boot = np.sqrt(errores_cuadrados[remuestreos].mean(axis=1))
    return float(np.percentile(rmses_boot, 2.5)), float(np.percentile(rmses_boot, 97.5))


filas_ci = []
for (year, familia), errores in errores_por_torneo_familia.items():
    ci_low, ci_high = bootstrap_ci_rmse(errores)
    filas_ci.append({"mundial": year, "familia": familia, "rmse_ci_low": ci_low, "rmse_ci_high": ci_high})

df_ci = pd.DataFrame(filas_ci)
df_comparacion = df_comparacion.merge(df_ci, on=["mundial", "familia"])
print(df_comparacion.round(4).to_string(index=False))

 mundial  familia   rmse  logloss  rmse_ci_low  rmse_ci_high
    2010      glm 1.2097   1.0206       1.0184        1.4013
    2010 lightgbm 1.0834   0.9821       0.8878        1.2964
    2010  xgboost 1.0847   0.9717       0.9049        1.2724
    2014      glm 1.2624   1.0180       1.0652        1.4677
    2014 lightgbm 1.2253   0.9712       1.0152        1.4439
    2014  xgboost 1.2346   0.9764       1.0288        1.4455
    2018      glm 1.2086   1.0425       1.0168        1.4140
    2018 lightgbm 1.1621   1.0371       0.9505        1.3785
    2018  xgboost 1.1830   1.0371       0.9707        1.4005
    2022      glm 1.3389   1.1251       1.0971        1.5906
    2022 lightgbm 1.2530   1.0619       1.0186        1.5107
    2022  xgboost 1.2512   1.0613       1.0143        1.5034
    2026      glm 1.2574   0.8771       1.0813        1.4313
    2026 lightgbm 1.2977   0.8931       1.1097        1.4831
    2026  xgboost 1.2812   0.8715       1.0949        1.4616


## 3.6 Vista agregada: ¿quién gana de verdad, a través de los 5 torneos?

RMSE medio, LogLoss medio y nº de veces que cada familia queda primera -- la respuesta que un
solo torneo no puede dar. Esta es la comparación que de verdad debe decidir qué familia usa
el Notebook 4, no el resultado de 2026 en solitario.

In [7]:
resumen = df_comparacion.groupby("familia").agg(
    rmse_medio=("rmse", "mean"), rmse_std=("rmse", "std"),
    logloss_medio=("logloss", "mean"),
).reset_index()
victorias = df_comparacion.loc[df_comparacion.groupby("mundial")["rmse"].idxmin(), "familia"].value_counts()
resumen["veces_gana"] = resumen["familia"].map(victorias).fillna(0).astype(int)
resumen = resumen.sort_values("rmse_medio").reset_index(drop=True)

print("=== RMSE/LogLoss medios a través de 5 Mundiales (2010, 2014, 2018, 2022, 2026) ===")
print(resumen.round(4).to_string(index=False))
print()
print(df_comparacion.pivot(index="mundial", columns="familia", values="rmse").round(4).to_string())

familia_recomendada = resumen.iloc[0]["familia"]
print(f"\nFamilia recomendada para el Notebook 4: {familia_recomendada}")

=== RMSE/LogLoss medios a través de 5 Mundiales (2010, 2014, 2018, 2022, 2026) ===
 familia  rmse_medio  rmse_std  logloss_medio  veces_gana
lightgbm      1.2043    0.0836         0.9891           3
 xgboost      1.2069    0.0771         0.9836           1
     glm      1.2554    0.0532         1.0167           1

familia     glm  lightgbm  xgboost
mundial                           
2010     1.2097    1.0834   1.0847
2014     1.2624    1.2253   1.2346
2018     1.2086    1.1621   1.1830
2022     1.3389    1.2530   1.2512
2026     1.2574    1.2977   1.2812

Familia recomendada para el Notebook 4: lightgbm


## 3.7 Persistencia

In [8]:
ruta_comparacion = DIR_RESULTS / "comparacion_modelos.csv"
df_comparacion.to_csv(ruta_comparacion, index=False)
print(f"Guardado: {ruta_comparacion}")

ruta_resumen = DIR_RESULTS / "comparacion_resumen.csv"
resumen.to_csv(ruta_resumen, index=False)
print(f"Guardado: {ruta_resumen}")

Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/results/comparacion_modelos.csv
Guardado: /Users/danielcanteragomez/portfolio/wc-2026-match-predictor/results/comparacion_resumen.csv


## Resumen

- Se comparan GLM, LightGBM y XGBoost (todos afinados con Optuna, incluido el `alpha` del
  GLM) sobre los 5 Mundiales de la era moderna disponibles: 2010, 2014, 2018, 2022 y 2026 --
  entrenando cada vez solo con datos anteriores al inicio de ese torneo concreto.
- `results/comparacion_modelos.csv`: RMSE/LogLoss + intervalo de confianza bootstrap, por
  Mundial y familia.
- `results/comparacion_resumen.csv`: RMSE medio, LogLoss medio y nº de victorias por familia
  a través de los 5 Mundiales -- **esto, no el resultado de un solo torneo, es lo que debe
  decidir la familia que usa el Notebook 4** (`nombre_ganador`).
- Random Forest queda fuera (ver intro): ya había perdido con claridad en una comparación
  previa y es, con diferencia, el más lento de tunear.

### Probado y descartado: features de SofaScore (posesión, tiros, corners, faltas)

Tras añadir en el Notebook 2 (sección 2.2b) las 8 features nuevas de SofaScore, se repitió
esta misma comparación de 5 Mundiales con `FEATURES_MODELO` ampliado -- resultado en
`results/comparacion_modelos_con_sofascore.csv` / `comparacion_resumen_con_sofascore.csv`.
**No se adoptan**: con ellas, el RMSE medio de XGBoost empeora (1.2201 -> 1.2229) y su
LogLoss también (0.9971 -> 1.0015); más revelador aún, la ventaja de XGBoost sobre LightGBM
-- que sin estas features gana 3 de 5 Mundiales con claridad -- se diluye hasta un empate
2-2. Con solo ~15% de cobertura real (el resto queda relleno con la media global, por ser
una fuente que solo cubre las 48 selecciones del Mundial 2026), añaden más ruido que señal
neta. Mismo criterio que decidió XGBoost sobre GLM: se mide, no se asume -- y aquí la
medida dice que no ayudan. El Notebook 4 sigue usando el `FEATURES_MODELO` original.

Se probó además una variante del cálculo de estas ventanas que SALTA los partidos sin estadística real (en vez de dejar que el hueco consuma posición de ventana y rellenarlo con la media global), medida solo contra 2026 por ser donde la cobertura de fondo es alta: tampoco ayuda, y en LogLoss es la peor de las tres variantes en las tres familias (`results/experimento_sofascore_dropna_2026.csv`) -- la hipótesis de que el problema fuera el relleno queda descartada; el problema es la cobertura de la fuente en sí.

### Probado y adoptado: calibración isotónica de las probabilidades 1X2

El Notebook 4 (4.10) solo diagnosticaba la calibración con un diagrama, nunca la
corregía. Se implementó la corrección real (`IsotonicRegression` uno-contra-el-resto por
clase, ajustada sobre predicciones out-of-fold con `TimeSeriesSplit`) y se validó -- no
solo en los 5 Mundiales, sino en un backtest bastante más amplio: **70 ediciones de 7
torneos distintos (Mundial, Eurocopa, Copa América, Copa Africana, Copa Asiática, Gold
Cup, Confederations Cup, 1990-2026), 1790 partidos** -- porque 5 Mundiales resultaron ser
una muestra demasiado pequeña para varias de estas decisiones (ver el hallazgo de
decaimiento exponencial, justo debajo).

| | Accuracy | LogLoss | Brier |
|---|--:|--:|--:|
| Sin calibrar | 54.13% | 0.9725 | 0.5791 |
| **Con calibración** | **54.92%** | **0.9722** | **0.5785** |

Mejora consistente en las 3 métricas, aunque modesta (a nivel de edición individual, solo
gana en LogLoss en 36/70 -- básicamente un empate partido a partido, la ventaja está en el
agregado). **Se adopta** y ya está integrada en el Notebook 4.

### Probado y descartado: ventanas de forma con decaimiento exponencial de recencia

Sustituir `dif_forma_gf_5`/`dif_forma_gf_10` (ventana fija) por una media ponderada con
decaimiento exponencial por tiempo (`ewm(halflife=...)`, 90/180/365 días) parecía mejorar
RMSE y LogLoss cuando solo se probaba contra los 5 Mundiales (~0.3-0.5%). **Con las 70
ediciones no queda ninguna mejora real** -- las tres variantes quedan indistinguibles del
baseline, y con halflife largo incluso empeora la accuracy. Es el ejemplo más claro de
esta ronda de por qué 5 Mundiales (266 partidos) es una muestra insuficiente para decidir
features: lo que parecía señal real era ruido de muestra pequeña. **No se adopta.**

### Probado, evidencia insuficiente: valor de mercado de la plantilla (Transfermarkt)

Añadir `diff_valor_mercado` (valor de mercado actual, scrapeado de Transfermarkt por el
propio usuario -- ver `scripts/scrape_transfermarkt_valor.py`, robots.txt prohíbe
crawlers de IA en ese dominio) mejoró las 3 métricas sobre la fase de grupos 2026 (RMSE
1.2974→1.2689, accuracy 61.1%→62.5%, LogLoss 0.9011→0.8729). **Pero es un dato ESTÁTICO**
(solo tenemos el valor de hoy, Transfermarkt no expone un histórico agregado por
selección) -- no se puede backtestear en 2010/2014/2018/2022 sin fuga temporal (usar el
valor de 2026 para "predecir" un partido de 2010 sería literalmente eso), así que la
evidencia se reduce a un solo torneo de 72 partidos -- exactamente el tipo de muestra que
el hallazgo de arriba (decaimiento exponencial) acaba de demostrar que no es fiable para
decidir. **No se adopta en producción** por falta de evidencia suficiente, no porque el
resultado en sí fuera malo -- es una limitación de disponibilidad de datos, no de método.
Si en el futuro aparece una fuente con histórico real por fecha, merece la pena
revisitarlo con el mismo rigor de 70 ediciones.

### Probado y adoptado: features de palmarés reciente

`dif_titulos_8a`, `dif_titulos_4a`, `dif_finales_8a` (títulos/finales de torneo mayor en los 8/4 años previos, derivados del propio `results.csv` + `shootouts.csv`) y `dif_ppp_vs_top_3a` (puntos por partido contra rivales de Elo ≥ 1900, 3 años). En el backtest de 5 Mundiales mejora RMSE, LogLoss y accuracy en las 3 familias (`results/experimento_palmares_5wc.csv`); confirmado sobre **82 ediciones / 2.677 partidos** de los 7 torneos mayores 1990-2026: LogLoss 0.9752→0.9736 (gana en 48/82 ediciones), accuracy 53.98%→54.28% (`results/verificacion_70ediciones.csv`). **Se adopta** — primera incorporación de features que supera el estándar completo desde la calibración isotónica.

### Probado y adoptado (solo para predecir Mundial): shrinkage de contexto de fase final

Las fases finales de Mundial son más cerradas que la mezcla de entrenamiento (2.57 vs 2.76 goles/partido) y el modelo sobreestima las lambdas. Corrección: multiplicarlas por `g = media_real / media_predicha`, estimado SOLO sobre Mundiales anteriores al torneo evaluado (g≈0.91-0.99 según el año). En 5 Mundiales sube el marcador exacto de 10.1% a **12.1%** y el top-3 de 33.5% a 36.4%, sin empeorar nunca ni tocar el 1X2 (`results/experimento_marcador_exacto_5wc.csv`). En el agregado de TODOS los torneos es neutro, así que **se aplica únicamente al predecir fase final de Mundial** (el caso de producción del Notebook 4). La variante adicional de recalibración multiplicativa de marcadores frecuentes se probó y **se descartó**: ruidosa (empeora 2026) y no añade nada sobre el shrinkage.

### Probado y descartado: regla de umbral para predecir empates

Predecir EMPATE cuando P(X) ≥ umbral (ajustado sobre Mundiales previos) en vez del argmax. Sube el recall de empates de 0% a 23%, pero con precisión insuficiente (8-38% según el torneo): la accuracy agregada cae de 53.1% a 48.9% (`results/experimento_empates_5wc.csv`). **No se adopta para accuracy** — con estas probabilidades, maximizar aciertos implica no predecir empates nunca. Nota para el módulo de apuestas: la precisión de los años buenos (31-38%) supera el break-even de las cuotas típicas de empate (~30%), así que la señal podría ser rentable como apuesta de valor aunque sea inútil para accuracy.